# Repointage Univers UNV → UNX

Notebook de migration des fournisseurs de données Webi d'un univers **UNV** vers son équivalent **UNX**.

> **Contexte** : conversion d'univers via SAP Information Design Tool (IDT).  
> **API utilisée** : SAP BusinessObjects 4.3 — REST API Raylight `/dataproviders/mappings`

### Pré-requis
- Le projet **SAPPy** doit être cloné localement
- Ce notebook doit être lancé depuis le dossier `SAPPy/notebooks/`
- La méthode `set_repoint_universe` doit être ajoutée dans `lib/webi.py` (voir section 0)

### Workflow
1. Initialisation de l'environnement SAPPy
2. Connexion à la plateforme BO
3. **[Optionnel]** Inspection du document et de ses fournisseurs de données
4. Calcul du mapping automatique (GET) + commit (POST) + sauvegarde CMS
5. Vérification post-repointage
6. Déconnexion

## 0. Intégration de `set_repoint_universe` dans `webi.py`

Exécutez la cellule ci-dessous **une seule fois** pour injecter automatiquement la nouvelle méthode dans `lib/webi.py`.  
Elle vérifie que la méthode n'est pas déjà présente avant d'agir.

> Le code source de la méthode est dans `lib/webi_repoint_method.py`.

In [1]:
APPLICATION_NAME = "WebiRepoint"

In [ ]:
import sys
from pathlib import Path

# Remonter les dossiers jusqu'à trouver lib/init/init_code.py
def find_app_home(sentinel: str ="lib/bootstrap/bootstrap.py"):
    current = Path.cwd().resolve()
    root = current.root
    while current != root:
        if (current / sentinel).is_file():
            return current
        current = current.parent
    raise FileNotFoundError(f"Impossible de trouver le fichier sentinelle : {sentinel}")

# Trouver et ajouter APPLICATION_HOME au sys.path
APPLICATION_HOME = find_app_home()
sys.path.insert(0, str(APPLICATION_HOME))
#print(f"APPLICATION_HOME set to: {APPLICATION_HOME}")

from lib.bootstrap.bootstrap import init_env
epy = init_env()

## 1. Initialisation de l'environnement SAPPy

In [3]:
props     = epy.cfgprops
log       = epy.log

url       = props.get('bo_url')
account   = props.get('bo_account')
password  = props.get('bo_password')
type_auth = props.get('bo_authentication')

In [4]:
bip  = epy.load_class(module_name='bip',  args=[APPLICATION_HOME, APPLICATION_NAME, url])
webi = epy.load_class(module_name='webi', args=[APPLICATION_HOME, APPLICATION_NAME, bip])

In [5]:
import time

## 2. Paramètres de la migration

| Paramètre | Valeur |
|---|---|
| Document Webi | **812407** |
| Univers source (UNV) | ID `7407` — CUID `B.3PSqZLENO2CSboMbVawN8` |
| Univers cible (UNX) | ID `812457` — CUID `AdVE6T_6B3JOkjT5tkhYsBw` |

In [7]:
# -----------------------------------------------
# Paramètres de la migration
# -----------------------------------------------
DOCUMENT_ID = epy.cfgyaml.get("documents_list")
UNV_ID: int = int(epy.cfgyaml.get("unv_id"))
UNV_CUID = epy.cfgyaml.get("unv_cuid")
UNX_ID: int = int(epy.cfgyaml.get("unx_id"))          # targetDatasourceId attendu par l'API
UNX_CUID = epy.cfgyaml.get("unx_cuid")
# Si True : commite même si des objets ne sont pas mappés automatiquement
FORCE_COMMIT : bool = False

In [8]:
start = time.time()

In [ ]:
log.log("#####################################################")
log.log("### REPOINTAGE DES DOCUMENTS WEBI SUR UNIVERS UNX ###")
log.log("#####################################################")
log.log("")

## 3. Connexion à la plateforme

In [ ]:
log.info(f"## Authentification sur la plateforme '{url}' ##")

In [ ]:
token = bip.set_token(base_url=url, username=account, password=password, auth_type=type_auth)
if token:
    # log.info("Authentification réussie")
    log.info(f'Token obtenu : {token[:40]}...')
else:
    log.error("Echec d'authentication")
log.log("")


## 4. Inspection du document *(optionnel)*

Vérification avant repointage : informations générales et état des fournisseurs de données.

In [ ]:
log.info(f"## Inspection des documents WebI ##")
log.log("")

In [ ]:
# Informations générales du document
# info = webi.get_doc_info(DOCUMENT_ID)
info_list = []
info_dp_list = []

for id in DOCUMENT_ID:
    info = webi.get_doc_info(id)
    log.log(f"Nom     : {info.get('name')} (id: {info.get('id')})")
    log.log(f"   Chemin  : {info.get('path')}")
    log.log(f"   Modifié : {info.get('updated')}")
    log.log(f"   Auteur  : {info.get('lastAuthor')}")
    info_list.append(info)

    # details des fournisseurs de données
    dp_list = webi.get_doc_dp(id, simplified=True)
    # info_dp_list.append(dp_list)
    log.log(f'   Nombre de DP : {len(dp_list)}')
    for dp in dp_list:
        info_dp_list.append(dp)
        log.log(f"     id={dp['id']:<8} type={dp.get('dataSourceType','?'):<6} "
            f"dataSourceId={dp.get('dataSourceId','?'):<12} name={dp.get('name')}")
    log.log("")


In [ ]:
log.info(f"## Identification des fournisseurs de données elligibles au repointage ##")
log.log("")

In [15]:
# print(type(info_dp_list))
# for t in info_dp_list:
#     print(t)

In [ ]:
# Identification des DP éligibles au repointage (type=unv)
dp_eligibles = [
    dp for dp in info_dp_list
    if str(dp.get('dataSourceId')) in (str(UNV_ID), UNV_CUID) and dp.get('dataSourceType', '').lower() == 'unv'
    # dp for dp in dp_list if (dp.get('dataSourceType', '').lower() == 'unv' )
]
log.log(f'DP éligibles ({len(dp_eligibles)}) :')
for dp in dp_eligibles:
    log.log(f"  → {dp['id']} | {dp.get('name')} | dataSourceId={dp.get('dataSourceId')}")
log.log("")

## 5. Repointage UNV → UNX

La méthode `set_repoint_universe` exécute pour chaque DP :

| Étape | Endpoint | Description |
|---|---|---|
| 1 | `GET /dataproviders/mappings?originDataproviderIds=DP0&targetDatasourceId=812457` | Calcul mapping automatique |
| 2 | *(vérification interne)* | Contrôle des status `Ok` / `Unresolved` |
| 3 | `POST /dataproviders/mappings` (body = XML du GET) | Commit du repointage |
| 4 | `PUT /documents/812407` (body vide) | Sauvegarde CMS |

In [ ]:
log.info(f"## Repointage UNV -> UNX ##")
log.info(f"Univers {UNV_ID} -> {UNX_ID} ")
log.log("")

In [ ]:
for id in DOCUMENT_ID:
    result = webi.set_repoint_universe(
        doc_id             = id,
        target_universe_id = UNX_ID,
        dp_ids             = None,       # None = auto-détection des DP unv/unx
        force              = FORCE_COMMIT
    )
log.log("")

## 6. Résultat et vérification

In [ ]:
log.info(f"## Résultat et vérification")
log.log("")

In [ ]:
log.log('=' * 55)
log.log(f"  Succès global  : {result['success']}")
log.log(f"  DP repointés   : {result['committed']}")
log.log(f"  DP ignorés     : {result['skipped']}")
log.log(f"  DP en erreur   : {result['errors']}")
log.log('=' * 55)
log.log("")

log.log('Détail mapping par DP :')
for dp_id, details in result['mapping_details'].items():
    unresolv = details['unresolved']
    log.log(f"  DP {dp_id} — total={details['total']} | ok={details['ok']} | "
          f"unresolved={len(unresolv)} {unresolv if unresolv else ''}")
log.log("")


In [ ]:
# Vérification post-repointage
for id in DOCUMENT_ID:
    dp_list_after = webi.get_doc_dp(id, simplified=True)
    log.info('Status des DP après repointage :')
    for dp in dp_list_after:
        ok = str(dp.get('dataSourceId')) in (str(UNX_ID), UNX_CUID)
        icon = '✅' if ok else '⚠'
        log.log(f"  {icon} id={dp['id']:<8} type={dp.get('dataSourceType','?'):<6} "
            f"dataSourceId={dp.get('dataSourceId','?'):<12} name={dp.get('name')}")
log.log("")

### En cas d'objets `Unresolved`

Si des DP sont dans `skipped` (mapping incomplet) :
1. Identifiez les objets non résolus dans `result['mapping_details']`
2. Vérifiez que ces objets existent bien dans l'UNX (via IDT)
3. Relancez avec `FORCE_COMMIT = True` pour forcer le commit même avec des objets non résolus *(utilisation au risque de l'administrateur)*
4. Ou corrigez l'UNX pour que les objets manquants soient présents

## 7. Déconnexion

In [ ]:
# Deconnexion
log.info("## Deconnexion ##")
bip.unset_token()
log.log("")
log.info(f"### Traitement terminée en {time.time() - start:.1f} secondes ###")